In [1]:
import pandas as pd
import numpy as np
import os

print("Environment ready.")

Environment ready.


In [2]:
print("Loading clean datasets and GTFS metadata...")
df_ridership = pd.read_csv('../data/processed/clean_ridership.csv')
df_stops = pd.read_csv('../data/processed/clean_stops.csv')
df_trips = pd.read_csv('../data/raw/trips.txt', usecols=['route_id', 'trip_id'])

# Ensure route IDs are string types for consistent merging
df_ridership['route'] = df_ridership['route'].astype(str)
df_trips['route_id'] = df_trips['route_id'].astype(str)
df_stops['stop_id'] = df_stops['stop_id'].astype(int)

print("Ridership records:", len(df_ridership))
print("Stops records:", len(df_stops))

Loading clean datasets and GTFS metadata...
Ridership records: 119826
Stops records: 11177


In [3]:
print("Building Spatial Grid Zones (500m x 500m)...")
# Rounding latitude/longitude to 0.005 degrees maps to roughly a 500m grid
grid_size = 0.005
df_stops['zone_lat'] = np.round(df_stops['stop_lat'] / grid_size) * grid_size
df_stops['zone_lon'] = np.round(df_stops['stop_lon'] / grid_size) * grid_size
df_stops['zone_id'] = df_stops.apply(lambda r: f"ZONE_{r['zone_lat']:.4f}_{r['zone_lon']:.4f}", axis=1)

print("Unique zones built:", df_stops['zone_id'].nunique())
df_stops.head()

Building Spatial Grid Zones (500m x 500m)...
Unique zones built: 1894


,stop_id,stop_name,stop_lat,stop_lon,zone_lat,zone_lon,zone_id
0,1,Jackson & Austin Terminal,41.876330,-87.774111,41.875,-87.775,ZONE_41.8750_-87.7750
1,2,5900 W Jackson,41.877075,-87.771324,41.875,-87.770,ZONE_41.8750_-87.7700
2,4,5700 W Jackson,41.876992,-87.768264,41.875,-87.770,ZONE_41.8750_-87.7700
3,6,Jackson & Lotus,41.876521,-87.761452,41.875,-87.760,ZONE_41.8750_-87.7600
4,7,5351 W Jackson,41.876560,-87.758931,41.875,-87.760,ZONE_41.8750_-87.7600


In [4]:
print("Mapping routes to stops using GTFS trip files...")
# Load stop_times in chunks for high-speed, memory-efficient loading
chunks = pd.read_csv('../data/raw/stop_times.txt', usecols=['trip_id', 'stop_id'], chunksize=100000)
df_stop_times = pd.concat([c for c in chunks], ignore_index=True)

# Link route_id to stop_id via trip_id
df_route_stops = pd.merge(df_trips, df_stop_times, on='trip_id')[['route_id', 'stop_id']].drop_duplicates()
df_route_stops['stop_id'] = df_route_stops['stop_id'].astype(int)

# Merge stop coordinates and zone IDs onto the route-stop mapping
df_route_zones = pd.merge(df_route_stops, df_stops, on='stop_id')
print("Mapped route-stop-zone records:", len(df_route_zones))
df_route_zones.head()

Mapping routes to stops using GTFS trip files...
Mapped route-stop-zone records: 14419


,route_id,stop_id,stop_name,stop_lat,stop_lon,zone_lat,zone_lon,zone_id
0,1,13155,Desplaines & Harrison Terminal,41.874451,-87.644351,41.875,-87.645,ZONE_41.8750_-87.6450
1,1,18661,Jefferson & Harrison,41.874611,-87.642345,41.875,-87.640,ZONE_41.8750_-87.6400
2,1,18498,Jefferson & Van Buren,41.876912,-87.642421,41.875,-87.640,ZONE_41.8750_-87.6400
3,1,67,Jackson & Canal,41.878005,-87.639782,41.880,-87.640,ZONE_41.8800_-87.6400
4,1,14461,Jackson & Chicago River,41.878012,-87.638312,41.880,-87.640,ZONE_41.8800_-87.6400


In [5]:
print("Selecting top 10 busiest routes to build a high-density forecasting dataset...")
top_routes = df_ridership.groupby('route')['rides'].sum().nlargest(10).index.tolist()
print("Top 10 Routes:", top_routes)

# Filter ridership and spatial mapping to the top routes
df_ridership_sub = df_ridership[df_ridership['route'].isin(top_routes)].copy()
df_route_zones_sub = df_route_zones[df_route_zones['route_id'].isin(top_routes)].copy()

# Count unique zones visited by each route to distribute ridership proportionally
zones_per_route = df_route_zones_sub.groupby('route_id')['zone_id'].nunique().to_dict()

# Merge ridership volumes with geographic zones
df_daily_zones = pd.merge(df_ridership_sub, df_route_zones_sub[['route_id', 'zone_id']].drop_duplicates(), 
                           left_on='route', right_on='route_id')

# Distribute route volume equally across all zones it intersects
df_daily_zones['zone_rides'] = df_daily_zones['rides'] / df_daily_zones['route_id'].map(zones_per_route)

# Sum ridership from all intersecting routes per zone and date
df_zone_daily = df_daily_zones.groupby(['zone_id', 'date', 'daytype'])['zone_rides'].sum().reset_index(name='daily_demand')
print("Zone-level daily records:", len(df_zone_daily))
df_zone_daily.head()

Selecting top 10 busiest routes to build a high-density forecasting dataset...
Top 10 Routes: ['79', '66', '9', '4', '8', '53', '77', '22', '49', '82']
Zone-level daily records: 499776


,zone_id,date,daytype,daily_demand
0,ZONE_41.6850_-87.6100,2019-01-01,U,137.928571
1,ZONE_41.6850_-87.6100,2019-01-02,W,270.607143
2,ZONE_41.6850_-87.6100,2019-01-03,W,308.821429
3,ZONE_41.6850_-87.6100,2019-01-04,W,314.339286
4,ZONE_41.6850_-87.6100,2019-01-05,A,221.982143


In [6]:
print("Performing Temporal Disaggregation (Daily -> Hourly) using vectorised curves...")
# 1. Define standard weekday and weekend hourly distribution profiles (summing to 1.0)
weekday_profile = [
    0.005, 0.002, 0.001, 0.002, 0.005, 0.020,
    0.060, 0.120, 0.100, 0.050, 0.040, 0.040,
    0.040, 0.040, 0.050, 0.070, 0.120, 0.100,
    0.060, 0.040, 0.020, 0.010, 0.005, 0.005
]
weekend_profile = [
    0.010, 0.005, 0.002, 0.002, 0.005, 0.010,
    0.020, 0.040, 0.060, 0.070, 0.080, 0.080,
    0.080, 0.080, 0.070, 0.060, 0.050, 0.040,
    0.030, 0.020, 0.015, 0.010, 0.008, 0.003
]

# 2. Vectorised cross-join to expand each zone-day record to 24 hours
hours_df = pd.DataFrame({'hour': range(24)})
df_hourly = df_zone_daily.merge(hours_df, how='cross')

# 3. Map hourly profiles using numpy vectorised conditions
weekday_weights = np.array(weekday_profile)
weekend_weights = np.array(weekend_profile)

df_hourly['weight'] = np.where(
    df_hourly['daytype'] == 'W',
    df_hourly['hour'].map(lambda h: weekday_weights[h]),
    df_hourly['hour'].map(lambda h: weekend_weights[h])
)

# 4. Calculate final hourly demand and create standard features
df_hourly['demand'] = np.round(df_hourly['daily_demand'] * df_hourly['weight'], 2)
df_hourly['is_weekend'] = df_hourly['daytype'].isin(['A', 'U']).astype(int)

# Clean up columns
df_hourly = df_hourly.drop(columns=['weight', 'daily_demand'])
print("Engineered Hourly Zone Demand records:", len(df_hourly))
df_hourly.head(25)

Performing Temporal Disaggregation (Daily -> Hourly) using vectorised curves...
Engineered Hourly Zone Demand records: 11994624


,zone_id,date,daytype,hour,demand,is_weekend
0,ZONE_41.6850_-87.6100,2019-01-01,U,0,1.38,1
1,ZONE_41.6850_-87.6100,2019-01-01,U,1,0.69,1
2,ZONE_41.6850_-87.6100,2019-01-01,U,2,0.28,1
3,ZONE_41.6850_-87.6100,2019-01-01,U,3,0.28,1
4,ZONE_41.6850_-87.6100,2019-01-01,U,4,0.69,1
5,ZONE_41.6850_-87.6100,2019-01-01,U,5,1.38,1
6,ZONE_41.6850_-87.6100,2019-01-01,U,6,2.76,1
7,ZONE_41.6850_-87.6100,2019-01-01,U,7,5.52,1
8,ZONE_41.6850_-87.6100,2019-01-01,U,8,8.28,1
9,ZONE_41.6850_-87.6100,2019-01-01,U,9,9.65,1


In [7]:
# Save feature engineered dataset to processed directory
df_hourly.to_csv('../data/processed/hourly_zone_demand.csv', index=False)
print("Feature-engineered hourly zone dataset saved successfully in /data/processed/")

Feature-engineered hourly zone dataset saved successfully in /data/processed/
